<a href="https://colab.research.google.com/github/krushna1133/NNDL-Practical/blob/main/NNDL_Practical9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =====================================================
# IMDB SENTIMENT ANALYSIS USING RNN
# =====================================================

# 1. Install kagglehub if needed
!pip install -q kagglehub

# 2. Import libraries
import kagglehub
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from sklearn.model_selection import train_test_split


# =====================================================
# 3. Download IMDB Dataset from Kaggle
# =====================================================

path = kagglehub.dataset_download("rehanliaqat17/imbd-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'imbd-dataset' dataset.
Path to dataset files: /kaggle/input/imbd-dataset


In [7]:


# =====================================================
# 4. Load Dataset
# =====================================================

import os

print("Files:", os.listdir(path))

# Usually the file is named IMDB Dataset.csv
df = pd.read_csv("/kaggle/input/imbd-dataset/IMDB_Dataset.csv")

print(df.head())
print("Dataset size:", df.shape)


# =====================================================
# 5. Convert Sentiment into 0 and 1
# =====================================================

df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})


# =====================================================
# 6. Split Dataset
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
)


# =====================================================
# 7. Convert Text into Numbers
# =====================================================

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, maxlen=200)
X_test = pad_sequences(X_test, maxlen=200)


# =====================================================
# 8. Create RNN Model
# =====================================================

model = Sequential([
    Embedding(10000, 32),
    SimpleRNN(32),
    Dense(1, activation="sigmoid")
])


# =====================================================
# 9. Compile Model
# =====================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# =====================================================
# 10. Train Model
# =====================================================

model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64
)


# =====================================================
# 11. Test Model
# =====================================================

loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy:", accuracy * 100, "%")


# =====================================================
# 12. Predict New Sentence
# =====================================================

def predict_sentiment(sentence):

    # Convert sentence into numbers
    sequence = tokenizer.texts_to_sequences([sentence])

    # Make length 200
    sequence = pad_sequences(sequence, maxlen=200)

    # Predict
    prediction = model.predict(sequence, verbose=0)[0][0]

    if prediction >= 0.5:
        print("\nPositive Review")
        print("The sentence has a positive sentiment.")
        print("Confidence:", round(prediction * 100, 2), "%")
    else:
        print("\nNegative Review")
        print("The sentence has a negative sentiment.")
        print("Confidence:", round((1 - prediction) * 100, 2), "%")


# =====================================================
# 13. Enter New Sentence
# =====================================================

sentence = input("\nEnter a movie review: ")

predict_sentiment(sentence)


Files: ['IMDB_Dataset.csv']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Dataset size: (50000, 2)
Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 26ms/step - accuracy: 0.7153 - loss: 0.5525
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.8363 - loss: 0.3883
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.7990 - loss: 0.4459
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7150 - loss: 0.5545

Test Accuracy: 71.49999737739563 %

Enter a movie review: This movie was fantastic and I really enjoyed it

Positive Review
The sentence has a positive sentiment.
Confidence: 77.31 %
